# 🌟 COSMIC v0.0.1 - Compatibilidad Automática

**Este notebook ha sido actualizado para COSMIC v0.0.1**

## ✨ Mejoras Disponibles
- **Estructura modular**: `cosmic.core`, `cosmic.io`, `cosmic.preprocess`, `cosmic.analysis`
- **Imports modernos**: Específicos y organizados
- **API mejorada**: Mejor manejo de errores y configuración
- **Compatibilidad total**: El código existente sigue funcionando

## 🔄 Migración Opcional
Para usar las nuevas funcionalidades:
```python
# Nuevo (recomendado)
from cosmic.analysis.analyzer import ClusterAnalyzer
from cosmic.core.clustering import Clustering
from cosmic.io.loader import DataLoader

# Legacy (sigue funcionando)
from COSMIC import ClusterAnalyzer
from clustering import HDBSCANClustering
from data_loader import DataLoader
```

📚 **Documentación**: `/docs/reference/NOTEBOOK_MIGRATION_GUIDE.md`

In [ ]:
from astropy.io import ascii
from astropy.table import QTable,join,vstack
from astropy.coordinates import SkyCoord, Galactocentric, Angle
from astropy.visualization import quantity_support
import hdbscan
import seaborn as sns
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from zero_point import zpt
import matplotlib.ticker as ticker
from astropy.coordinates import SkyCoord
from astropy.table import QTable, hstack
zpt.load_tables()
quantity_support()
%matplotlib inline
%config InlineBackend.figure_format ='retina'

In [ ]:
data_gaia = QTable.read('GAIA_2MASS_WISE_40ARCMIN.ecsv',guess=False,format='ascii.ecsv')

In [ ]:
for column_name in data_gaia.colnames:
    if hasattr(data_gaia[column_name], 'mask'):  # Check if the column has a mask attribute
        # Convert column to float if it's an integer type
        if np.issubdtype(data_gaia[column_name].dtype, np.integer):
            data_gaia[column_name] = data_gaia[column_name].astype(float)
        # Now safe to fill masked values with np.nan
        data_gaia[column_name] = data_gaia[column_name].filled(np.nan)

num_fidelity_numeric = np.sum(np.isfinite(data_gaia['fidelity_v2']))
total_sources = len(data_gaia)
print(f'The data contains {total_sources} sources')
if total_sources == num_fidelity_numeric:
    print('All the sources have fidelity')
else:
    print(f'Only {num_fidelity_numeric} sources have fidelity')
print(f"There's {np.sum(~np.isnan(data_gaia['tmass_oid']))} of {total_sources} with 2MASS data")
print(f"There's {np.sum(~np.isnan(data_gaia['r_med_geo']))} of {total_sources} with geometric distances")

In [ ]:
# # Edit table columns
data_gaia.rename_columns(['phot_g_mean_mag', 'phot_bp_mean_mag', 'phot_rp_mean_mag', 'bp_rp'],['Gmag', 'G_BPmag', 'G_RPmag', 'BP_RP'])

In [ ]:
columns_to_check = ['ra','dec','pmra','pmdec','parallax','Gmag','G_BPmag','G_RPmag']
mask = np.zeros(len(data_gaia), dtype=bool)
for column in columns_to_check:
    mask |= ~np.isfinite(data_gaia[column])

# Apply the mask to the QTable
data_gaia = data_gaia[~mask]

In [ ]:
print(len(data_gaia))

In [ ]:
# # Zero-Point Parallax
data_gaia.rename_columns(['parallax'],['parallax_observed']);
data_gaia['zpvals'] = zpt.get_zpt(data_gaia['Gmag'], data_gaia['nu_eff_used_in_astrometry'], data_gaia['pseudocolour'], data_gaia['ecl_lat'], data_gaia['astrometric_params_solved'])*u.mas
data_gaia['zpvals'] = np.ma.masked_invalid(data_gaia['zpvals']).filled(0)
data_gaia['parallax'] = data_gaia['parallax_observed'] - data_gaia['zpvals']

In [ ]:
data_gaia['pmra_obs'],data_gaia['pmdec_obs'] = data_gaia['pmra'],data_gaia['pmdec']

In [ ]:
# Correct proper motion to align with ICRF
def edr3ToICRF(pmra ,pmdec ,ra ,dec ,G):
    if G >=13:
        return pmra , pmdec
    def sind (x):
        return np.sin(np. radians (x))
    def cosd (x):
        return np.cos(np. radians (x))
    table1 =""" 0.0 9.0 18.4 33.8 -11.3
                9.0 9.5 14.0 30.7 -19.4
                9.5 10.0 12.8 31.4 -11.8
                10.0 10.5 13.6 35.7 -10.5
                10.5 11.0 16.2 50.0 2.1
                11.0 11.5 19.4 59.9 0.2
                11.5 11.75 21.8 64.2 1.0
                11.75 12.0 17.7 65.6 -1.9
                12.0 12.25 21.3 74.8 2.1
                12.25 12.5 25.7 73.6 1.0
                12.5 12.75 27.3 76.6 0.5
                12.75 13.0 34.9 68.9 -2.9 """
    table1 = np.fromstring(table1,sep=' ').reshape((12,5)).T
    Gmin = table1[0]
    Gmax = table1[1]
    # pick the appropriate omegaXYZ for the source ’s magnitude :
    omegaX = table1[2][(Gmin <=G)&(Gmax>G)][0]
    omegaY = table1[3][(Gmin <=G)&(Gmax>G)][0]
    omegaZ = table1[4][(Gmin <=G)&(Gmax>G)][0]
    pmraCorr = -1*sind(dec)*cosd(ra)*omegaX-sind(dec)*sind(ra)*omegaY + cosd(dec)*omegaZ
    pmdecCorr = sind(ra)*omegaX-cosd(ra)*omegaY
    return pmra - pmraCorr/1000. , pmdec - pmdecCorr/1000.

In [ ]:
for i in data_gaia:
    i['pmra'],i['pmdec'] = edr3ToICRF(i['pmra_obs'].value,i['pmdec_obs'].value,i['ra'].value,i['dec'].value,i['Gmag'].value)*(u.mas/u.yr)

In [ ]:
def add_photometric_errors(table):
    """
    Adds photometric errors for Gmag, G_BPmag, G_RPmag, and e_bp_rp to the given QTable,
    incorporating astropy units (u.mag) for proper handling of magnitudes, with errors expressed in milli-magnitudes.

    Parameters:
    - table: QTable, must contain columns 'Gmag', 'G_BPmag', 'G_RPmag', and their respective fluxes and flux errors.

    The function modifies the table in place by adding four new columns for the errors:
    - 'e_Gmag', 'e_G_BPmag', 'e_G_RPmag', 'e_BP_RP', all expressed in magnitudes.
    """

    # Function to calculate magnitude error from flux and flux error
    def calculate_mag_error(flux, flux_error):
        # Ensure that the units are correct (e.g., electrons per second)
        flux = flux.value * (u.electron / u.s)
        flux_error = flux_error.value * (u.electron / u.s)
        return 2.5 * np.log10(1 + (flux_error / flux)) * u.mag

    # Compute the magnitude errors using the flux and flux errors
    table['e_Gmag'] = calculate_mag_error(table['phot_g_mean_flux'], table['phot_g_mean_flux_error'])
    table['e_G_BPmag'] = calculate_mag_error(table['phot_bp_mean_flux'], table['phot_bp_mean_flux_error'])
    table['e_G_RPmag'] = calculate_mag_error(table['phot_rp_mean_flux'], table['phot_rp_mean_flux_error'])
    
    # Calculate e_BP_RP as the square root of the sum of squares of e_G_BPmag and e_G_RPmag
    table['e_BP_RP'] = np.sqrt(table['e_G_BPmag']**2 + table['e_G_RPmag']**2)
    
    # If you have other columns such as 'j_msigcom', 'h_msigcom', and 'ks_msigcom', you can compute their errors similarly
    if 'j_msigcom' in table.colnames and 'h_msigcom' in table.colnames and 'ks_msigcom' in table.colnames:
        table['e_J_H'] = np.sqrt(table['j_msigcom']**2 + table['h_msigcom']**2)
        table['e_RP_J'] = np.sqrt(table['e_G_RPmag']**2 + table['j_msigcom']**2)
        table['e_H_K'] = np.sqrt(table['h_msigcom']**2 + table['ks_msigcom']**2)
        table['e_BP_J'] = np.sqrt(table['e_G_BPmag']**2 + table['j_msigcom']**2)

In [ ]:
add_photometric_errors(data_gaia)

In [ ]:
# Existing color indices with TMASS
data_gaia['RP_J'] = data_gaia['G_RPmag'] - data_gaia['j_m']
data_gaia['H_K'] = data_gaia['h_m'] - data_gaia['ks_m']
data_gaia['J_K'] = data_gaia['j_m'] - data_gaia['ks_m']
data_gaia['J_H'] = data_gaia['j_m'] - data_gaia['h_m']
data_gaia['BP_J'] = data_gaia['G_BPmag'] - data_gaia['j_m']

# New color indices combining Gaia, TMASS, and WISE
data_gaia['RP_W1'] = data_gaia['G_RPmag'] - data_gaia['w1mpro']
data_gaia['RP_W2'] = data_gaia['G_RPmag'] - data_gaia['w2mpro']
data_gaia['RP_W3'] = data_gaia['G_RPmag'] - data_gaia['w3mpro']
data_gaia['RP_W4'] = data_gaia['G_RPmag'] - data_gaia['w4mpro']

data_gaia['BP_W1'] = data_gaia['G_BPmag'] - data_gaia['w1mpro']
data_gaia['BP_W2'] = data_gaia['G_BPmag'] - data_gaia['w2mpro']
data_gaia['BP_W3'] = data_gaia['G_BPmag'] - data_gaia['w3mpro']
data_gaia['BP_W4'] = data_gaia['G_BPmag'] - data_gaia['w4mpro']

data_gaia['J_W1'] = data_gaia['j_m'] - data_gaia['w1mpro']
data_gaia['J_W2'] = data_gaia['j_m'] - data_gaia['w2mpro']
data_gaia['J_W3'] = data_gaia['j_m'] - data_gaia['w3mpro']
data_gaia['J_W4'] = data_gaia['j_m'] - data_gaia['w4mpro']

data_gaia['H_W1'] = data_gaia['h_m'] - data_gaia['w1mpro']
data_gaia['H_W2'] = data_gaia['h_m'] - data_gaia['w2mpro']
data_gaia['H_W3'] = data_gaia['h_m'] - data_gaia['w3mpro']
data_gaia['H_W4'] = data_gaia['h_m'] - data_gaia['w4mpro']

data_gaia['K_W1'] = data_gaia['ks_m'] - data_gaia['w1mpro']
data_gaia['K_W2'] = data_gaia['ks_m'] - data_gaia['w2mpro']
data_gaia['K_W3'] = data_gaia['ks_m'] - data_gaia['w3mpro']
data_gaia['K_W4'] = data_gaia['ks_m'] - data_gaia['w4mpro']

In [ ]:
data_gaia['r_med_geo'] = data_gaia['r_med_geo'].to(u.kpc)
data_gaia['r_lo_geo'] = data_gaia['r_lo_geo'].to(u.kpc)
data_gaia['r_hi_geo'] = data_gaia['r_hi_geo'].to(u.kpc)
data_gaia['r_med_photogeo'] = data_gaia['r_med_photogeo'].to(u.kpc)
data_gaia['r_lo_photogeo'] = data_gaia['r_lo_photogeo'].to(u.kpc)
data_gaia['r_hi_photogeo'] = data_gaia['r_hi_photogeo'].to(u.kpc)

In [ ]:
fidelity_selection = (0.5 < data_gaia['fidelity_v2'])
bad_gaia = data_gaia[~fidelity_selection]
data_gaia = data_gaia[fidelity_selection]
print(f"{len(data_gaia)} remaining, {round(len(data_gaia)/total_sources*100)}% of good sources, with {np.sum(~np.isnan(data_gaia['tmass_oid']))} sources with 2MASS data and {np.sum(~np.isnan(data_gaia['allwise_oid']))} sources with WISE data")

In [ ]:
ascii.write(data_gaia,'good_data.ecsv',format='ecsv',overwrite=True) # Create the clustered file

In [ ]:
ascii.write(bad_gaia,'bad_data.ecsv',format='ecsv',overwrite=True) # Create the clustered file

In [ ]:
n = []
past_len = np.inf
min_cluster_size_samples = np.arange(10, 300, 1)
effective_mcs = []
lambda_value = []
labels_storage = {i: [] for i in range(len(data_gaia))}  # Initialize storage for labels

for i in min_cluster_size_samples:
    clusterer = hdbscan.HDBSCAN(algorithm='best',
                                cluster_selection_method='leaf',
                                allow_single_cluster=True,
                                min_cluster_size=i,
                                core_dist_n_jobs=-1,
                                gen_min_span_tree=True,
                                metric='euclidean',
                                cluster_selection_persistence
                               ).fit(data_gaia['pmra', 'pmdec'].to_pandas())

    # Store the labels for each point
    for idx, label in enumerate(clusterer.labels_):
        labels_storage[idx].append(label)
    
    tree = clusterer.condensed_tree_.to_pandas()
    if (tree["lambda_val"].max() >= 8):
        max_lambda_val_row = tree["lambda_val"].idxmax()
        desired_parent = tree.at[max_lambda_val_row, "parent"]
        desired_len = len(tree[tree["parent"] == desired_parent])
        if (desired_len <= 1000) and (len(np.unique(clusterer.labels_)) > 1) and (desired_len >= 200):
            n.append(desired_len)
            effective_mcs.append(i)
            lambda_value.append(tree["lambda_val"].max())

In [ ]:
cluster_probabilities = np.array([np.count_nonzero(np.array(labels) != -1) / len(labels) for labels in labels_storage.values()])

In [ ]:
# Find the index of the maximum value in n for plotting the vertical line
max_n_value = max(n)
max_n_index = n.index(max_n_value)  # Guard against empty list
max_min_cluster_size = effective_mcs[max_n_index]

max_lambda_value = lambda_value[max_n_index]
max_lambda_index = lambda_value.index(max_lambda_value)
max_lambda_min_cluster_size = effective_mcs[max_lambda_index]

# Plotting
fig,ax = plt.subplots(layout='constrained',figsize=(6,5))
ax.plot(effective_mcs, n)
ax.axvline(max_min_cluster_size, color='b', linestyle='--', label=f'Optimal Min Cluster Size: {max_min_cluster_size}')
ax.axhline(y=max_n_value, color='g', linestyle='--', label=f'Max Cluster Size: {max_n_value}')
#ax.axvline(min_min_cluster_size, color='r', linestyle='--', label=f'Optimal Min Cluster Size: {min_min_cluster_size}')
#ax.axhline(y=min_n_value, color='fuchsia', linestyle='--', label=f'Max Cluster Size: {min_n_value}')
ax.set_xlabel('Min Cluster Size',fontsize=16)
ax.set_ylabel('Cluster Size',fontsize=16)
ax.legend()

fig.savefig('../Tex_File/Figures/min_cluster_size.pdf',bbox_inches='tight')
plt.show()

In [ ]:
clusterer = hdbscan.HDBSCAN(algorithm='best',
                            cluster_selection_method='leaf',
                            allow_single_cluster=True,
                            min_cluster_size=max_min_cluster_size,
                            core_dist_n_jobs=-1,
                            gen_min_span_tree=True,
                            metric='euclidean',
                           ).fit(data_gaia['pmra','pmdec'].to_pandas())

In [ ]:
tree = clusterer.condensed_tree_.to_pandas()
# Find the index of the row with the maximum lambda_val
max_lambda_val_row = tree["lambda_val"].idxmax()
# Retrieve the parent value for the row with the maximum lambda_val
desired_parent = tree.at[max_lambda_val_row, "parent"]
desired_len = len(tree[tree["parent"] == desired_parent])

In [ ]:
### Crea el condensed cluster tree.

fig_ct, ax_ct = plt.subplots(1,1, layout='constrained',figsize=(8,6))
clusterer.condensed_tree_.plot(select_clusters=True,selection_palette=sns.color_palette('bright',len(np.unique(clusterer.labels_))),cmap=sns.color_palette("mako", as_cmap=True),axis=ax_ct)
ax_ct.set_ylabel(r'$\lambda$ value',fontsize=16)
ax_ct.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_ct.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_ct.tick_params(axis='both', which='both', direction='in')
fig_ct.savefig('../Tex_File/Figures/condensed_cluster_tree_NGC6383.pdf');

In [ ]:
data_gaia['cluster'] = clusterer.labels_
data_gaia['probability_hdbscan'] = clusterer.probabilities_
data_gaia['probability_times'] = cluster_probabilities
data_gaia['probability'] = data_gaia['probability_hdbscan']*data_gaia['probability_times']
color_palette = sns.color_palette('bright',len(np.unique(clusterer.labels_)))
data_gaia['outlier_score'] = clusterer.outlier_scores_

In [ ]:
# Calculate the size of each cluster in data_gaia_qtable
# QTable does not have the value_counts method, so we'll use a different approach
clusters, counts = np.unique(data_gaia["cluster"], return_counts=True)

# Find the cluster(s) with the same size as the desired cluster size
matching_clusters = clusters[counts == desired_len]

# For simplicity, assuming there's only one matching cluster and getting its name
desired_cluster_name_qtable = matching_clusters[0] if len(matching_clusters) > 0 else None

desired_cluster_name_qtable, len(matching_clusters)  # Returning the cluster name and the number of matching clusters

In [ ]:
data_gaia['cluster_hdbscan'] = data_gaia['cluster']

In [ ]:
data_gaia['cluster'] = np.where(data_gaia['cluster'] == desired_cluster_name_qtable, 
                                       data_gaia['cluster'], 
                                       -1)

In [ ]:
combined_table = vstack([data_gaia,bad_gaia])
combined_table['cluster'] = combined_table['cluster'].filled(-1)
combined_table['probability_hdbscan'] = combined_table['probability_hdbscan'].filled(0)
combined_table['probability_times'] = combined_table['probability_times'].filled(0)
combined_table['probability'] = combined_table['probability'].filled(0)

In [ ]:
ascii.write(combined_table,'40_arcmin_clustered.ecsv',format='ecsv',overwrite=True) # Create the clustered file